In [89]:
import pandas as pd

In [90]:
df=pd.read_parquet("../data/clean/clean_olist_orders_dataset.parquet")

### Target variable

Target in the answer we want the model to learn.
We will be selecting Target variable as late_delivery because we will be predicting whether the item will get delivered before the estimated delivery date for not. And we will be doing this so by using the information available during the order processing lifecycle.

In [91]:
df['late_delivery']=(df['order_delivered_customer_date']>df['order_estimated_delivery_date']).astype(int)

In [92]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,late_delivery
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,0


### Class Distribution

In [93]:
df['late_delivery'].value_counts(normalize=True)

late_delivery
0    0.918869
1    0.081131
Name: proportion, dtype: float64

### Meaningful Feature

1. Approval Duration
- Orders that take longer to get approved may have a higher chance of being delivered late.
Measurement: Time between Purchase and Approval,
2. Expected Delivery Duration
- As we can calculculate expected number of days to deliver the order.
3. Purchage Day/Month
- Weekend orders and any season sale orders might get delayed because of large number of orders getting placed, so there might be delay in time to approve the order.
4. Time of Purchase (Day or Night)
- As from the orders placed and approval time we can see that order placed at the late night take more time to approve.

- Every feature satisfies two condition:
1. Available at prediction time (after approval)
2. Has business explanation

| Approval_Duration | Estimated_Shipping_Days | Purchase_Day | Purchase_Hour | Late_Delivery |
| ----------------- | ----------------------- | ------------ | ------------- | ------------- |
| 0.5               | 7                       | Monday       | 10            | 0             |
| 48                | 15                      | Sunday       | 23            | 1             |


In [94]:
df['purchase_hour']=df['order_purchase_timestamp'].dt.hour
df['purchase_hour'].head()

0    10
1    20
2     8
3    19
4    21
Name: purchase_hour, dtype: int32

In [95]:
df['purchase_day']=df['order_purchase_timestamp'].dt.day_name()
df['purchase_day'].head()

0       Monday
1      Tuesday
2    Wednesday
3     Saturday
4      Tuesday
Name: purchase_day, dtype: object

In [96]:
df['purchase_month']=df['order_purchase_timestamp'].dt.month
df['purchase_month'].head()

0    10
1     7
2     8
3    11
4     2
Name: purchase_month, dtype: int32

In [97]:
expected_duration=(df['order_estimated_delivery_date']-df['order_purchase_timestamp'])
df['expected_delivery_days']=expected_duration.dt.days.head()
df['expected_delivery_days'].head()

0    15.0
1    19.0
2    26.0
3    26.0
4    12.0
Name: expected_delivery_days, dtype: float64

In [98]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,late_delivery,purchase_hour,purchase_day,purchase_month,expected_delivery_days
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,0,10,Monday,10,15.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,0,20,Tuesday,7,19.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,0,8,Wednesday,8,26.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,0,19,Saturday,11,26.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,0,21,Tuesday,2,12.0


In [99]:
approval_duration=(df['order_approved_at']-df['order_purchase_timestamp'])
df['approval_hours']=(approval_duration.dt.total_seconds()/3600)
df['approval_hours'].head()

0     0.178333
1    30.713889
2     0.276111
3     0.298056
4     1.030556
Name: approval_hours, dtype: float64

In [100]:
df.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'late_delivery', 'purchase_hour', 'purchase_day', 'purchase_month',
       'expected_delivery_days', 'approval_hours'],
      dtype='object')

In [101]:
df['order_status'].value_counts()

order_status
delivered    96455
canceled         6
Name: count, dtype: int64

In [102]:
rem_cols=['order_id','customer_id','order_status','order_purchase_timestamp','order_approved_at',
'order_delivered_carrier_date',
'order_delivered_customer_date',
'order_estimated_delivery_date']
df=df.drop(rem_cols,axis=1)

In [103]:
df.head()

,late_delivery,purchase_hour,purchase_day,purchase_month,expected_delivery_days,approval_hours
0,0,10,Monday,10,15.0,0.178333
1,0,20,Tuesday,7,19.0,30.713889
2,0,8,Wednesday,8,26.0,0.276111
3,0,19,Saturday,11,26.0,0.298056
4,0,21,Tuesday,2,12.0,1.030556


In [104]:
dumm_days=pd.get_dummies(df.purchase_day,drop_first=True,).astype(int)

In [105]:
df=pd.concat([df,dumm_days],axis=1)
df.head()

,late_delivery,purchase_hour,purchase_day,purchase_month,expected_delivery_days,approval_hours,Monday,Saturday,Sunday,Thursday,Tuesday,Wednesday
0,0,10,Monday,10,15.0,0.178333,1,0,0,0,0,0
1,0,20,Tuesday,7,19.0,30.713889,0,0,0,0,1,0
2,0,8,Wednesday,8,26.0,0.276111,0,0,0,0,0,1
3,0,19,Saturday,11,26.0,0.298056,0,1,0,0,0,0
4,0,21,Tuesday,2,12.0,1.030556,0,0,0,0,1,0


In [107]:
df.drop('purchase_day',inplace=True,axis=1)

In [108]:
df.head()

,late_delivery,purchase_hour,purchase_month,expected_delivery_days,approval_hours,Monday,Saturday,Sunday,Thursday,Tuesday,Wednesday
0,0,10,10,15.0,0.178333,1,0,0,0,0,0
1,0,20,7,19.0,30.713889,0,0,0,0,1,0
2,0,8,8,26.0,0.276111,0,0,0,0,0,1
3,0,19,11,26.0,0.298056,0,1,0,0,0,0
4,0,21,2,12.0,1.030556,0,0,0,0,1,0


In [110]:
df.late_delivery.value_counts(normalize=True)

late_delivery
0    0.918869
1    0.081131
Name: proportion, dtype: float64